# LeetCode 1734 — Decode XORed Permutation bằng Python

Link đề: https://leetcode.com/problems/decode-xored-permutation/ · Độ khó: **Medium** · Chủ đề: Array, Bit Manipulation

Skeleton LeetCode:

```python
class Solution:
    def decode(self, encoded: List[int]) -> List[int]:
        ...
```

Lộ trình: **thử mọi giá trị cho `perm[0]`** rồi giải mã và kiểm tra hoán vị (`O(n²)`) → mẹo then chốt: `n` **lẻ** nên XOR các `encoded` ở vị trí **lẻ** cho `perm[1] ^ … ^ perm[n−1]`; kết hợp với `1 ^ 2 ^ … ^ n` để ra `perm[0]` (`O(n)`) → XOR `1..n` bằng **công thức chu kỳ 4** → góc nhìn khác qua **XOR tiền tố**.

## 1. Mục tiêu học

1. Nắm các tính chất của **XOR**: giao hoán, kết hợp, `x ^ x = 0`, `x ^ 0 = x` — và dùng chúng để "triệt tiêu" phần tử.
2. Giải mã dãy từ `encoded[i] = perm[i] ^ perm[i+1]` khi đã biết **một** phần tử: `perm[i+1] = perm[i] ^ encoded[i]`.
3. Tìm phần tử đầu tiên nhờ hai thông tin: **XOR toàn bộ hoán vị** (`1 ^ … ^ n`) và **XOR các `encoded` ở vị trí lẻ**; hiểu vai trò của điều kiện `n` **lẻ**.
4. Tính `1 ^ 2 ^ … ^ n` trong `O(1)` bằng chu kỳ `n mod 4`.
5. Kiểm chứng bằng cách sinh hoán vị ngẫu nhiên, mã hóa rồi giải mã.

## 2. Đề bài & ví dụ

Có một mảng số nguyên `perm` là **hoán vị** của `n` số nguyên dương đầu tiên (`1..n`), với `n` luôn **lẻ**.

Nó được mã hóa thành mảng `encoded` độ dài `n − 1` với `encoded[i] = perm[i] XOR perm[i + 1]`. Ví dụ `perm = [1,3,2]` ⇒ `encoded = [2,1]`.

Cho `encoded`, trả về mảng `perm` gốc. Đề bảo đảm đáp án tồn tại và duy nhất.

**Ví dụ 1:** `encoded = [3,1]` → `[1,2,3]` (vì `[1^2, 2^3] = [3, 1]`).

**Ví dụ 2:** `encoded = [6,5,4,6]` → `[2,4,1,5,3]`.

**Ràng buộc:**
- `3 ≤ n < 10⁵`, `n` lẻ
- `encoded.length == n − 1`

In [1]:
import random
import time
from functools import reduce
from operator import xor
from typing import List

examples = [
    ([3, 1], [1, 2, 3]),
    ([6, 5, 4, 6], [2, 4, 1, 5, 3]),
]
for enc, want in examples:
    print(f"encoded={enc} -> kỳ vọng {want}")

def encode(perm):
    return [perm[i] ^ perm[i + 1] for i in range(len(perm) - 1)]

print([encode(w) == e for e, w in examples])

encoded=[3, 1] -> kỳ vọng [1, 2, 3]
encoded=[6, 5, 4, 6] -> kỳ vọng [2, 4, 1, 5, 3]
[True, True]


## 3. Kiến thức nền

### 3.1 Toán tử XOR `^`

XOR từng bit: bit kết quả là 1 khi hai bit khác nhau. Tính chất: `a ^ a = 0`, `a ^ 0 = a`, giao hoán, kết hợp ⇒ trong một dãy XOR, mỗi giá trị xuất hiện **chẵn** lần sẽ biến mất.

In [2]:
a, b = 6, 3
print(f"{a:04b} ^ {b:04b} = {a ^ b:04b} ({a ^ b})")
print(5 ^ 5, 5 ^ 0, 1 ^ 2 ^ 1, 7 ^ 3 ^ 7 ^ 3)

0110 ^ 0011 = 0101 (5)
0 5 2 0


### 3.2 XOR có "nghịch đảo" là chính nó

Nếu `c = a ^ b` thì `a = c ^ b` và `b = c ^ a`. Đây là lý do giải mã chỉ cần **một** phần tử khởi đầu.

In [3]:
perm = [2, 4, 1, 5, 3]
enc = encode(perm)
dec = [perm[0]]
for e in enc:
    dec.append(dec[-1] ^ e)
print("encoded =", enc, "| giải mã từ perm[0]=2:", dec)

encoded = [6, 5, 4, 6] | giải mã từ perm[0]=2: [2, 4, 1, 5, 3]


### 3.3 `functools.reduce` với `operator.xor`

In [4]:
print(reduce(xor, [1, 2, 3, 4, 5]), reduce(xor, range(1, 6)), reduce(xor, [], 0))

1 1 0


### 3.4 XOR của `1..n` theo chu kỳ 4

`f(n) = 1 ^ 2 ^ … ^ n` có quy luật: `n % 4 == 0 → n`, `1 → 1`, `2 → n + 1`, `3 → 0`.

In [5]:
def xor_1_to_n(n):
    return [n, 1, n + 1, 0][n % 4]

print([xor_1_to_n(n) for n in range(1, 13)])
print([reduce(xor, range(1, n + 1)) for n in range(1, 13)])

[1, 3, 0, 4, 1, 7, 0, 8, 1, 11, 0, 12]
[1, 3, 0, 4, 1, 7, 0, 8, 1, 11, 0, 12]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2: `encoded = [6,5,4,6]`, `n = 5`, hoán vị chưa biết `[p0, p1, p2, p3, p4]`.

- `encoded[1] = p1 ^ p2`, `encoded[3] = p3 ^ p4` ⇒ `encoded[1] ^ encoded[3] = p1 ^ p2 ^ p3 ^ p4` (mọi phần tử **trừ** `p0`).
- `total = 1 ^ 2 ^ 3 ^ 4 ^ 5 = p0 ^ p1 ^ p2 ^ p3 ^ p4` (hoán vị chứa đúng các số `1..5`) `= 1`.
- `p0 = total ^ (encoded[1] ^ encoded[3]) = 1 ^ (5 ^ 6) = 1 ^ 3 = 2`. ✓

Rồi giải mã dây chuyền: `p1 = 2 ^ 6 = 4`, `p2 = 4 ^ 5 = 1`, `p3 = 1 ^ 4 = 5`, `p4 = 5 ^ 6 = 3` ⇒ `[2,4,1,5,3]`.

**Vì sao cần `n` lẻ?** `n − 1` chẵn nên các `encoded` ở vị trí lẻ `1, 3, …, n−2` ghép thành đúng `(n−1)/2` cặp phủ **kín** `p1..p_{n−1}`.

In [6]:
enc = [6, 5, 4, 6]; n = len(enc) + 1
total = reduce(xor, range(1, n + 1))
odd = reduce(xor, enc[1::2])
print(f"total = {total}, XOR encoded vị trí lẻ = {odd}, perm[0] = {total ^ odd}")

total = 1, XOR encoded vị trí lẻ = 3, perm[0] = 2


## 5. Cách 1 — Brute Force / baseline: thử mọi `perm[0]`

**Ý tưởng:** `perm[0]` là một số trong `1..n`. Với mỗi ứng viên, giải mã toàn bộ dãy và kiểm tra có phải hoán vị của `1..n` không. Đề bảo đảm đúng một ứng viên thành công.

In [7]:
def decode_brute(encoded):
    n = len(encoded) + 1
    for first in range(1, n + 1):
        perm = [first]
        for e in encoded:
            perm.append(perm[-1] ^ e)
        if sorted(perm) == list(range(1, n + 1)):
            return perm
    return []

for enc, want in examples:
    print(decode_brute(enc), "kỳ vọng", want)

[1, 2, 3] kỳ vọng [1, 2, 3]
[2, 4, 1, 5, 3] kỳ vọng [2, 4, 1, 5, 3]


Có thể kiểm tra nhanh hơn bằng mảng đánh dấu thay vì `sorted` và dừng sớm khi giá trị ra ngoài `[1, n]` hoặc trùng — nhưng worst case vẫn `O(n)` mỗi ứng viên.

**Độ phức tạp:** `n` ứng viên × `O(n log n)` (hoặc `O(n)` với mảng đánh dấu) ⇒ `O(n²)` trở lên. `n = 10⁵` ⇒ `10¹⁰` — quá chậm.

**Nút thắt:** ta cần **xác định trực tiếp** `perm[0]`.

In [8]:
def decode_brute_fast(encoded):
    """Brute force với dừng sớm khi giá trị ngoài [1, n] hoặc trùng."""
    n = len(encoded) + 1
    for first in range(1, n + 1):
        seen = [False] * (n + 1); x = first; ok = True
        seen[x] = True
        for e in encoded:
            x ^= e
            if x < 1 or x > n or seen[x]:
                ok = False; break
            seen[x] = True
        if ok:
            out = [first]
            for e in encoded: out.append(out[-1] ^ e)
            return out
    return []

random.seed(0)
for n in [1001, 3001]:
    perm = random.sample(range(1, n + 1), n); enc = encode(perm)
    t0 = time.perf_counter(); r = decode_brute_fast(enc)
    print(f"n={n}: đúng? {r == perm}, {time.perf_counter() - t0:.3f}s")

n=1001: đúng? True, 0.004s
n=3001: đúng? True, 0.007s


## 6. Ý tưởng thuật toán / bước đột phá

**Hai đẳng thức:**
1. `T = p0 ^ p1 ^ … ^ p_{n−1} = 1 ^ 2 ^ … ^ n` (vì `perm` là hoán vị của `1..n`).
2. `O = encoded[1] ^ encoded[3] ^ … ^ encoded[n−2] = (p1 ^ p2) ^ (p3 ^ p4) ^ … ^ (p_{n−2} ^ p_{n−1}) = p1 ^ … ^ p_{n−1}`.

⇒ `p0 = T ^ O`.

**Quy tắc giải mã:** `p_{i+1} = p_i ^ encoded[i]`.

**Biến thể đối xứng:** XOR các `encoded` ở vị trí **chẵn** `0, 2, …, n−3` cho `p0 ^ … ^ p_{n−2}` ⇒ `p_{n−1} = T ^ E`, rồi giải mã ngược từ cuối.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `perm[0] = (1^…^n) ^ (XOR encoded ở vị trí lẻ)`

In [9]:
def decode_xor(encoded):
    n = len(encoded) + 1
    total = 0
    for v in range(1, n + 1):
        total ^= v
    odd = 0
    for i in range(1, n - 1, 2):
        odd ^= encoded[i]
    perm = [total ^ odd]
    for e in encoded:
        perm.append(perm[-1] ^ e)
    return perm

for enc, want in examples:
    print(decode_xor(enc), "kỳ vọng", want)

[1, 2, 3] kỳ vọng [1, 2, 3]
[2, 4, 1, 5, 3] kỳ vọng [2, 4, 1, 5, 3]


**Độ phức tạp:** Time `O(n)` (ba vòng tuyến tính), Space `O(n)` cho kết quả (`O(1)` phụ).

### 7.2 Cách 3 — Công thức `O(1)` cho `1 ^ … ^ n` + slicing

In [10]:
def decode_formula(encoded):
    n = len(encoded) + 1
    total = [n, 1, n + 1, 0][n % 4]
    first = total ^ reduce(xor, encoded[1::2], 0)
    perm = [first]
    for e in encoded:
        perm.append(perm[-1] ^ e)
    return perm

for enc, want in examples:
    print(decode_formula(enc), "kỳ vọng", want)

[1, 2, 3] kỳ vọng [1, 2, 3]
[2, 4, 1, 5, 3] kỳ vọng [2, 4, 1, 5, 3]


Với `n` lẻ chỉ có hai trường hợp: `n % 4 == 1 ⇒ T = 1`, `n % 4 == 3 ⇒ T = 0`.

**Độ phức tạp:** Time `O(n)` (phần giải mã), Space `O(n)`.

### 7.3 Cách 4 — XOR tiền tố

Đặt `q[i] = encoded[0] ^ … ^ encoded[i−1]` (với `q[0] = 0`). Khi đó `q[i] = p0 ^ p_i` (các phần tử giữa triệt tiêu). XOR mọi `q[i]` với `i = 0..n−1`:

`q[0] ^ … ^ q[n−1] = (p0 ^ p0) ^ (p0 ^ p1) ^ … ^ (p0 ^ p_{n−1}) = p0^{(n lần)} ^ T = p0 ^ T` (vì `n` lẻ).

⇒ `p0 = T ^ (XOR mọi q[i])`, và `p_i = p0 ^ q[i]`.

In [11]:
def decode_prefix(encoded):
    n = len(encoded) + 1
    q = [0] * n
    for i, e in enumerate(encoded):
        q[i + 1] = q[i] ^ e
    total = [n, 1, n + 1, 0][n % 4]
    acc = 0
    for v in q:
        acc ^= v
    p0 = total ^ acc
    return [p0 ^ v for v in q]

for enc, want in examples:
    print(decode_prefix(enc), "kỳ vọng", want)

[1, 2, 3] kỳ vọng [1, 2, 3]
[2, 4, 1, 5, 3] kỳ vọng [2, 4, 1, 5, 3]


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 2 `encoded = [6,5,4,6]`:

| bước | phép tính | giá trị |
|---|---|---|
| T | `1^2^3^4^5` | 1 |
| O | `encoded[1] ^ encoded[3] = 5 ^ 6` | 3 |
| p0 | `T ^ O` | 2 |
| p1 | `2 ^ 6` | 4 |
| p2 | `4 ^ 5` | 1 |
| p3 | `1 ^ 4` | 5 |
| p4 | `5 ^ 6` | 3 |

In [12]:
def trace_xor(encoded):
    n = len(encoded) + 1
    total = 0
    for v in range(1, n + 1):
        total ^= v
    print(f"T = 1^...^{n} = {total} ({total:04b})")
    odd = 0
    for i in range(1, n - 1, 2):
        odd ^= encoded[i]
        print(f"  O ^= encoded[{i}]={encoded[i]:2d} ({encoded[i]:04b}) -> O = {odd} ({odd:04b})")
    p = total ^ odd
    print(f"perm[0] = T ^ O = {total} ^ {odd} = {p}")
    perm = [p]
    for i, e in enumerate(encoded):
        perm.append(perm[-1] ^ e)
        print(f"perm[{i + 1}] = perm[{i}] ^ encoded[{i}] = {perm[-2]} ^ {e} = {perm[-1]}")
    return perm

trace_xor([6, 5, 4, 6])

T = 1^...^5 = 1 (0001)
  O ^= encoded[1]= 5 (0101) -> O = 5 (0101)
  O ^= encoded[3]= 6 (0110) -> O = 3 (0011)
perm[0] = T ^ O = 1 ^ 3 = 2
perm[1] = perm[0] ^ encoded[0] = 2 ^ 6 = 4
perm[2] = perm[1] ^ encoded[1] = 4 ^ 5 = 1
perm[3] = perm[2] ^ encoded[2] = 1 ^ 4 = 5
perm[4] = perm[3] ^ encoded[3] = 5 ^ 6 = 3


[2, 4, 1, 5, 3]

In [13]:
def trace_prefix(encoded):
    n = len(encoded) + 1; q = [0] * n
    for i, e in enumerate(encoded):
        q[i + 1] = q[i] ^ e
    total = [n, 1, n + 1, 0][n % 4]; acc = reduce(xor, q, 0)
    print(f"q (= p0 ^ p_i) = {q}, XOR mọi q = {acc}, T = {total}, p0 = {total ^ acc}")
    print("perm =", [(total ^ acc) ^ v for v in q])

trace_prefix([6, 5, 4, 6])
trace_prefix([3, 1])

q (= p0 ^ p_i) = [0, 6, 3, 7, 1], XOR mọi q = 3, T = 1, p0 = 2
perm = [2, 4, 1, 5, 3]
q (= p0 ^ p_i) = [0, 3, 2], XOR mọi q = 1, T = 0, p0 = 1
perm = [1, 2, 3]


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def decode(self, encoded: List[int]) -> List[int]:
        n = len(encoded) + 1
        total = 1 if n % 4 == 1 else 0          # 1^2^...^n với n lẻ
        odd = 0
        for i in range(1, n - 1, 2):
            odd ^= encoded[i]
        perm = [total ^ odd]
        for e in encoded:
            perm.append(perm[-1] ^ e)
        return perm

sol = Solution()
for enc, want in examples:
    got = sol.decode(enc)
    print(got, "OK" if got == want else "SAI")

[1, 2, 3] OK
[2, 4, 1, 5, 3] OK


## 10. Vì sao thuật toán đúng

**(a) `T = p0 ^ … ^ p_{n−1}`.** XOR giao hoán và kết hợp, nên XOR các phần tử của một hoán vị không phụ thuộc thứ tự ⇒ bằng `1 ^ 2 ^ … ^ n`.

**(b) `O = p1 ^ … ^ p_{n−1}`.** Với `n` lẻ, các chỉ số lẻ `i = 1, 3, …, n − 2` có đúng `(n−1)/2` giá trị; `encoded[i] = p_i ^ p_{i+1}` phủ các cặp `(p1,p2), (p3,p4), …, (p_{n−2}, p_{n−1})` — rời nhau và hợp lại đúng `{p1, …, p_{n−1}}`.

**(c)** `T ^ O = p0 ^ (p1 ^ … ^ p_{n−1}) ^ (p1 ^ … ^ p_{n−1}) = p0`.

**(d) Giải mã.** `p_{i+1} = p_i ^ encoded[i]` vì `p_i ^ (p_i ^ p_{i+1}) = p_{i+1}`. Quy nạp cho mọi `i`.

**Công thức `1..n` theo chu kỳ 4:** với `k` chẵn, `k ^ (k+1) = 1`; nhóm theo khối `(4j, 4j+1, 4j+2, 4j+3)` cho XOR `0`. Từ đó suy ra bốn trường hợp. Với `n` lẻ: `n = 4j + 1 ⇒ T = 1`, `n = 4j + 3 ⇒ T = 0`.

**Tính duy nhất:** khi `perm[0]` cố định, cả dãy cố định ⇒ đáp án duy nhất (đề bảo đảm tồn tại).

## 11. Phân tích Time / Space

Gọi `n < 10⁵` (lẻ).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử `perm[0]` | `O(n²)` worst | `O(n)` | dừng sớm giúp nhiều thực tế |
| 2. XOR vị trí lẻ | `O(n)` | `O(n)` kết quả | **nộp bài** |
| 3. Công thức `1..n` | `O(n)` | `O(n)` | gọn |
| 4. XOR tiền tố | `O(n)` | `O(n)` | góc nhìn khác |

In [15]:
random.seed(1)
n = 99999
perm = random.sample(range(1, n + 1), n); enc = encode(perm)
for name, fn in [("xor", decode_xor), ("formula", decode_formula), ("prefix", decode_prefix), ("submit", Solution().decode)]:
    t0 = time.perf_counter(); r = fn(enc)
    print(f"{name:7s}: đúng? {r == perm}  {time.perf_counter() - t0:.4f}s")

xor    : đúng? True  0.0096s
formula: đúng? True  0.0075s
prefix : đúng? True  0.0121s
submit : đúng? True  0.0057s


## 12. Bộ test

Sinh hoán vị ngẫu nhiên với `n` **lẻ** `≥ 3`, mã hóa, giải mã, so sánh.

In [16]:
fns = [decode_brute, decode_brute_fast, decode_xor, decode_formula, decode_prefix, lambda e: Solution().decode(e)]

for enc, want in examples:
    for fn in fns:
        assert fn(enc) == want

edge = [[1, 2, 3], [3, 2, 1], [2, 1, 3], [1, 3, 2]]
for perm in edge:
    for fn in fns:
        assert fn(encode(perm)) == perm, (fn, perm)

random.seed(2024)
for _ in range(1000):
    n = random.choice(range(3, 60, 2))
    perm = random.sample(range(1, n + 1), n)
    enc = encode(perm)
    for fn in fns:
        assert fn(enc) == perm, (fn, perm)
print("ví dụ + edge + 1000 hoán vị ngẫu nhiên: giải mã đúng")

ví dụ + edge + 1000 hoán vị ngẫu nhiên: giải mã đúng


In [17]:
# kiểm tra tính duy nhất (đề bảo đảm): với n lẻ chỉ có đúng một perm[0] hợp lệ
random.seed(5)
for _ in range(200):
    n = random.choice(range(3, 40, 2)); perm = random.sample(range(1, n + 1), n); enc = encode(perm)
    valid = []
    for first in range(1, n + 1):
        p = [first]
        for e in enc: p.append(p[-1] ^ e)
        if sorted(p) == list(range(1, n + 1)): valid.append(first)
    assert valid == [perm[0]]
print("với n lẻ: luôn đúng một giá trị perm[0] hợp lệ (200 bộ)")

# với n chẵn, có thể có nhiều hơn một nghiệm -> giải thích vì sao đề yêu cầu n lẻ
def valid_firsts(encoded):
    n = len(encoded) + 1; out = []
    for first in range(1, n + 1):
        p = [first]
        for e in encoded: p.append(p[-1] ^ e)
        if sorted(p) == list(range(1, n + 1)): out.append(p)
    return out

for perm in ([1, 2, 3, 4], [1, 3, 2, 4], [2, 3, 4, 5, 6, 1]):
    print(f"n={len(perm)} (chẵn), perm={perm}: các hoán vị cùng encoded = {valid_firsts(encode(perm))}")

với n lẻ: luôn đúng một giá trị perm[0] hợp lệ (200 bộ)
n=4 (chẵn), perm=[1, 2, 3, 4]: các hoán vị cùng encoded = [[1, 2, 3, 4]]
n=4 (chẵn), perm=[1, 3, 2, 4]: các hoán vị cùng encoded = [[1, 3, 2, 4]]
n=6 (chẵn), perm=[2, 3, 4, 5, 6, 1]: các hoán vị cùng encoded = [[2, 3, 4, 5, 6, 1], [5, 4, 3, 2, 1, 6]]


Output cho thấy với `n = 6` (chẵn), hai hoán vị `[2,3,4,5,6,1]` và `[5,4,3,2,1,6]` có **cùng** `encoded`: hoán vị thứ hai là hoán vị thứ nhất XOR từng phần tử với `7` (XOR cùng một hằng số không đổi các `encoded`), và phép `x ↦ x ^ 7` biến tập `{1..6}` thành chính nó. Vậy với một số `n` chẵn đáp án **không duy nhất**; ngoài ra mẹo "XOR vị trí lẻ" cũng không phủ kín `p1..p_{n−1}` khi `n − 1` lẻ. Đây là lý do đề yêu cầu `n` lẻ.

## 13. Lỗi người mới hay gặp

### Lỗi 1 — XOR các `encoded` ở vị trí **chẵn** rồi coi là `p1 ^ … ^ p_{n−1}`

In [18]:
def wrong_even_positions(encoded):
    n = len(encoded) + 1
    total = reduce(xor, range(1, n + 1))
    even = reduce(xor, encoded[0::2], 0)           # SAI: đây là p0 ^ ... ^ p_{n-2}
    perm = [total ^ even]
    for e in encoded:
        perm.append(perm[-1] ^ e)
    return perm

for enc, want in examples:
    print("sai =", wrong_even_positions(enc), "| đúng =", want)

sai = [3, 0, 1] | đúng = [1, 2, 3]
sai = [3, 5, 0, 4, 2] | đúng = [2, 4, 1, 5, 3]


`T ^ E` cho ra `p_{n−1}` (phần tử **cuối**), không phải `p0`. **Sửa:** dùng vị trí lẻ cho `p0`, hoặc dùng vị trí chẵn nhưng giải mã **ngược** từ cuối.

### Lỗi 2 — Lấy XOR **mọi** phần tử `encoded`

In [19]:
def wrong_all_encoded(encoded):
    n = len(encoded) + 1
    total = reduce(xor, range(1, n + 1))
    allx = reduce(xor, encoded, 0)                 # SAI: = p0 ^ p_{n-1}
    perm = [total ^ allx]
    for e in encoded:
        perm.append(perm[-1] ^ e)
    return perm

for enc, want in examples:
    print("sai =", wrong_all_encoded(enc), "| đúng =", want)

sai = [2, 1, 0] | đúng = [1, 2, 3]
sai = [0, 6, 3, 7, 1] | đúng = [2, 4, 1, 5, 3]


XOR mọi `encoded` triệt tiêu dây chuyền chỉ còn `p0 ^ p_{n−1}` — không dùng trực tiếp được. **Sửa:** chỉ lấy vị trí lẻ.

### Lỗi 3 — Công thức `1 ^ … ^ n` sai chu kỳ (ví dụ nhầm `n % 4 == 1 → n`)

In [20]:
def wrong_cycle(n):
    return [n, n, n + 1, 0][n % 4]                # SAI: n % 4 == 1 phải ra 1

print("n=5: sai =", wrong_cycle(5), "| đúng =", reduce(xor, range(1, 6)))
print("n=9: sai =", wrong_cycle(9), "| đúng =", reduce(xor, range(1, 10)))

n=5: sai = 5 | đúng = 1
n=9: sai = 9 | đúng = 1


**Sửa:** `[n, 1, n + 1, 0][n % 4]`; khi nghi ngờ, so với `reduce(xor, range(1, n + 1))` cho vài `n`.

### Lỗi 4 — Dùng `n = len(encoded)` (quên `+ 1`)

In [21]:
def wrong_n(encoded):
    n = len(encoded)                               # SAI: n = len(encoded) + 1
    total = reduce(xor, range(1, n + 1))
    odd = reduce(xor, encoded[1::2], 0)
    perm = [total ^ odd]
    for e in encoded:
        perm.append(perm[-1] ^ e)
    return perm

for enc, want in examples:
    print("sai =", wrong_n(enc), "| đúng =", want)

sai = [2, 1, 0] | đúng = [1, 2, 3]
sai = [7, 1, 4, 0, 6] | đúng = [2, 4, 1, 5, 3]


**Sửa:** `n = len(encoded) + 1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mảng XOR kề nhau + biết thêm một ràng buộc toàn cục" ⇒ tìm **một** phần tử, rồi giải mã dây chuyền.
- XOR **các vị trí cách nhau 2** để triệt tiêu xen kẽ — mẹo tương tự với tổng xen dấu.
- `1 ^ … ^ n` ⇒ chu kỳ 4.
- Điều kiện lạ trong đề (`n` lẻ) thường là **gợi ý** cho lời giải.
- Họ bài: 1720 *Decode XORed Array* (đã cho `first`), 1310 *XOR Queries of a Subarray* (XOR tiền tố), 268 *Missing Number* (XOR `0..n`).

## 15. Biến thể của bài

1. **LeetCode 1720 — đã cho `first`:** chỉ cần giải mã dây chuyền.
2. **Mã hóa với khoảng cách 2:** `encoded[i] = perm[i] ^ perm[i+2]` — tách thành hai dây chuyền chẵn/lẻ.
3. **`perm` là hoán vị của `0..n−1`:** đổi `T` thành `0 ^ 1 ^ … ^ (n−1)`.
4. **`n` chẵn:** mẹo không còn áp dụng; có thể có nhiều `perm[0]` hợp lệ — phải thử (Cách 1).
5. **Mã hóa bằng tổng `perm[i] + perm[i+1]`:** cùng ý tưởng xen kẽ (tổng xen dấu thay cho XOR).

In [22]:
def decode_sum(encoded_sum):
    """Biến thể 5: encoded[i] = perm[i] + perm[i+1], perm là hoán vị 1..n, n lẻ."""
    n = len(encoded_sum) + 1
    T = n * (n + 1) // 2
    O = sum(encoded_sum[1::2])                 # p1 + ... + p_{n-1}
    p = [T - O]
    for e in encoded_sum:
        p.append(e - p[-1])
    return p

def decode_zero_based(encoded):
    """Biến thể 3: hoán vị của 0..n-1."""
    n = len(encoded) + 1
    T = reduce(xor, range(n), 0)
    p = [T ^ reduce(xor, encoded[1::2], 0)]
    for e in encoded:
        p.append(p[-1] ^ e)
    return p

random.seed(3)
for _ in range(200):
    n = random.choice(range(3, 30, 2)); perm = random.sample(range(1, n + 1), n)
    assert decode_sum([perm[i] + perm[i + 1] for i in range(n - 1)]) == perm
    perm0 = [x - 1 for x in perm]
    assert decode_zero_based(encode(perm0)) == perm0
print("biến thể tổng và hoán vị 0..n-1: OK")

biến thể tổng và hoán vị 0..n-1: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử `perm[0]` | giải mã + kiểm tra hoán vị | `O(n²)` | `O(n)` | kiểm chứng, `n` chẵn |
| 2. XOR vị trí lẻ | `T ^ O` | `O(n)` | `O(n)` | **nộp bài** |
| 3. Công thức chu kỳ 4 | `T` trong `O(1)` | `O(n)` | `O(n)` | gọn nhất |
| 4. XOR tiền tố | `p0 = T ^ ⊕q` | `O(n)` | `O(n)` | khi đã có sẵn prefix |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `decode([2, 1])` và `decode([1, 2, 3, 4])`?

**Bài 2 (tính tay):** `encoded = [7, 5, 4, 5, 7, 5]` (`n = 7`): tính `T`, `O`, `perm[0]` rồi giải mã.

**Bài 3 (điền TODO):** hoàn thiện `decode_from_last(encoded)` — tìm `perm[n−1]` bằng XOR các `encoded` ở vị trí **chẵn**, rồi giải mã ngược.

**Bài 4 (tự viết lại):** viết `decode_gap2(encoded2)` cho biến thể `encoded2[i] = perm[i] ^ perm[i+2]` (độ dài `n − 2`), biết thêm `perm[0]` và `perm[1]`.

**Bài 5 (mini challenge):** với `n` **chẵn**, đếm số hoán vị `perm` của `1..n` có cùng `encoded` với một hoán vị cho trước. Thực nghiệm với `n = 4, 6, 8` (vét cạn) và đưa ra nhận xét.

In [23]:
def decode_from_last(encoded):
    # TODO: E = encoded[0] ^ encoded[2] ^ ... ; last = T ^ E; giải mã ngược perm[i] = perm[i+1] ^ encoded[i]
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `[2,1]`: `n = 3`, `T = 0`, `O = encoded[1] = 1` ⇒ `p0 = 1`. (2) `[1, 3, 2]`. (3) `[1,2,3,4]`: `n = 5`, `T = 1`, `O = 2 ^ 4 = 6` ⇒ `p0 = 7` > 5?! — input này **không** phải mã hóa của hoán vị nào (đề bảo đảm input hợp lệ; hãy tự kiểm tra bằng Cách 1).

**Bài 2:** (1) `T = 1^…^7 = 0` (vì `7 % 4 == 3`). (2) `O = encoded[1] ^ encoded[3] ^ encoded[5] = 5 ^ 5 ^ 5 = 5`. (3) `p0 = 5` ⇒ giải mã `[5, 2, 7, 3, 6, 1, 4]`.

**Bài 3:** (1) `E` gồm `encoded[0], encoded[2], …, encoded[n−3]`. (2) `last = T ^ E`. (3) Duyệt `i` từ `n−2` về 0.

**Bài 4:** (1) Hai dây chuyền: chỉ số chẵn từ `perm[0]`, chỉ số lẻ từ `perm[1]`. (2) `perm[i+2] = perm[i] ^ encoded2[i]`. (3) Điền theo `i` tăng dần.

**Bài 5:** (1) Với `encoded` cố định, `perm` xác định bởi `perm[0]`. (2) Đếm số `first ∈ [1, n]` cho dãy là hoán vị. (3) So sánh các `n`.

## 19. Đáp án tham khảo

In [24]:
# Bài 1
assert Solution().decode([2, 1]) == [1, 3, 2]
print("decode_brute([1,2,3,4]) =", decode_brute([1, 2, 3, 4]), "(rỗng = không có hoán vị nào ứng với input này)")

# Bài 2
perm7 = [5, 2, 7, 3, 6, 1, 4]
assert encode(perm7) == [7, 5, 4, 5, 7, 5] and Solution().decode([7, 5, 4, 5, 7, 5]) == perm7

# Bài 3
def decode_from_last_ans(encoded):
    n = len(encoded) + 1
    T = [n, 1, n + 1, 0][n % 4]
    E = reduce(xor, encoded[0::2], 0)
    perm = [0] * n; perm[-1] = T ^ E
    for i in range(n - 2, -1, -1):
        perm[i] = perm[i + 1] ^ encoded[i]
    return perm

# Bài 4
def decode_gap2(encoded2, p0, p1):
    n = len(encoded2) + 2; perm = [0] * n; perm[0], perm[1] = p0, p1
    for i in range(n - 2):
        perm[i + 2] = perm[i] ^ encoded2[i]
    return perm

random.seed(101)
for _ in range(300):
    n = random.choice(range(3, 50, 2)); perm = random.sample(range(1, n + 1), n)
    assert decode_from_last_ans(encode(perm)) == perm
    enc2 = [perm[i] ^ perm[i + 2] for i in range(n - 2)]
    assert decode_gap2(enc2, perm[0], perm[1]) == perm
print("Bài 1-4: OK")

decode_brute([1,2,3,4]) = [] (rỗng = không có hoán vị nào ứng với input này)
Bài 1-4: OK


In [25]:
# Bài 5 — n chẵn: có bao nhiêu hoán vị chung một encoded?
from itertools import permutations
from collections import Counter

for n in [4, 6, 8]:
    groups = Counter(tuple(encode(list(p))) for p in permutations(range(1, n + 1)))
    dist = Counter(groups.values())
    print(f"n={n}: số encoded khác nhau = {len(groups)}, phân bố 'số hoán vị chung một encoded' = {dict(sorted(dist.items()))}")

n = 5
groups5 = Counter(tuple(encode(list(p))) for p in permutations(range(1, n + 1)))
print("n=5 (lẻ): phân bố =", dict(Counter(groups5.values())))

n=4: số encoded khác nhau = 24, phân bố 'số hoán vị chung một encoded' = {1: 24}
n=6: số encoded khác nhau = 360, phân bố 'số hoán vị chung một encoded' = {2: 360}
n=8: số encoded khác nhau = 40320, phân bố 'số hoán vị chung một encoded' = {1: 40320}
n=5 (lẻ): phân bố = {1: 120}


Nhận xét Bài 5 (từ output đã chạy): với `n = 4` và `n = 8` mỗi `encoded` vẫn ứng với đúng một hoán vị, nhưng với `n = 6` **mọi** `encoded` ứng với đúng **hai** hoán vị. Lý do: hai hoán vị cùng `encoded` ⇔ chúng khác nhau bởi phép XOR mọi phần tử với một hằng `x ≠ 0`, và cần `x ↦ v ^ x` biến `{1..n}` thành chính nó. Với `n = 6`, `x = 7` thỏa (`{1..6} ^ 7 = {6..1}`); tổng quát điều này xảy ra khi `n = 2^k − 2`. Với `n` lẻ thì không có `x` như vậy (XOR mọi phần tử của `{1..n}` với `x` phải giữ nguyên `T`, nhưng thay đổi `T` đi `x` vì số phần tử lẻ) — phân bố toàn là `1`, khớp với đảm bảo "đáp án duy nhất" của đề.

## 20. Template nhớ nhanh + Checklist

```python
n = len(encoded) + 1
total = [n, 1, n + 1, 0][n % 4]                # 1 ^ 2 ^ ... ^ n
odd = 0
for i in range(1, n - 1, 2):                   # encoded ở vị trí LẺ = p1 ^ ... ^ p_{n-1}
    odd ^= encoded[i]
perm = [total ^ odd]                           # p0
for e in encoded:
    perm.append(perm[-1] ^ e)                  # p_{i+1} = p_i ^ encoded[i]
return perm
```

**Checklist trước khi submit:**
- [ ] `n = len(encoded) + 1`.
- [ ] XOR các `encoded` ở vị trí **lẻ** (`1, 3, …`) để ra `p1 ^ … ^ p_{n−1}`.
- [ ] `T = 1 ^ … ^ n` đúng chu kỳ 4 (với `n` lẻ: `1` nếu `n % 4 == 1`, `0` nếu `n % 4 == 3`).
- [ ] Giải mã bằng `^`, không phải `-`.
- [ ] Thử `n = 3`.